# 08  Real-World Validation
## ShopEase Europe | Sentiment Analysis Project 
**Objective:** Validate our trained models against genuine customer 
reviews from a real-world multilingual dataset, addressing the 
template-based limitation identified in the ShopEase dataset. This 
notebook tests whether our models generalise beyond the synthetic 
training data.

In [4]:
import os
import pandas as pd
from datasets import load_dataset
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded successfully")

Libraries loaded successfully


## Download Real-World Sample
Pulling a small sample of genuine customer reviews across our four 
core languages from the Amazon Multilingual Reviews dataset on 
Hugging Face.

In [9]:
languages = ['en', 'fr', 'de', 'es']
samples = []

for lang in languages:
    dataset = load_dataset('mteb/amazon_reviews_multi', lang, split='validation')
    dataset = dataset.shuffle(seed=42)
    lang_samples = dataset.select(range(20))
    for item in lang_samples:
        samples.append({
            'language': lang,
            'review_text': item['text'],
            'star_rating': item['label'] + 1
        })

real_world_df = pd.DataFrame(samples)
print(f"Downloaded {len(real_world_df)} real reviews across {len(languages)} languages")
print(f"\nStar rating distribution:")
print(real_world_df['star_rating'].value_counts().sort_index())

Found cached dataset amazon_reviews_multi (C:/Users/User/.cache/huggingface/datasets/mteb___amazon_reviews_multi/en/1.0.0/61154b14772a2d73f3554caa8f5bd1fec4b65ec64f70bb93fa9fec6b038a4355)
Loading cached shuffled indices for dataset at C:\Users\User\.cache\huggingface\datasets\mteb___amazon_reviews_multi\en\1.0.0\61154b14772a2d73f3554caa8f5bd1fec4b65ec64f70bb93fa9fec6b038a4355\cache-7f3c2b000fc47742.arrow
Found cached dataset amazon_reviews_multi (C:/Users/User/.cache/huggingface/datasets/mteb___amazon_reviews_multi/fr/1.0.0/61154b14772a2d73f3554caa8f5bd1fec4b65ec64f70bb93fa9fec6b038a4355)
Loading cached shuffled indices for dataset at C:\Users\User\.cache\huggingface\datasets\mteb___amazon_reviews_multi\fr\1.0.0\61154b14772a2d73f3554caa8f5bd1fec4b65ec64f70bb93fa9fec6b038a4355\cache-dec31a132c0e1c2c.arrow
Found cached dataset amazon_reviews_multi (C:/Users/User/.cache/huggingface/datasets/mteb___amazon_reviews_multi/de/1.0.0/61154b14772a2d73f3554caa8f5bd1fec4b65ec64f70bb93fa9fec6b038a43

Downloaded 80 real reviews across 4 languages

Star rating distribution:
star_rating
1    20
2    12
3    16
4    24
5     8
Name: count, dtype: int64


## Initial Sampling Issue

Two initial download attempts using streaming mode returned entirely 
negative reviews. The first attempt (60 reviews, no shuffle) and a 
second attempt adding `.shuffle()` before `.take()` (80 reviews) both 
returned only 1-star ratings across all four languages, despite the 
dataset documentation confirming a balanced 5-label distribution. The 
issue and resolution are detailed below


## Sampling Method Correction

Initial attempts using streaming mode with `.take()` after shuffling 
returned exclusively 1-star reviews across all languages, despite the 
underlying dataset being balanced across 5 rating levels. This was 
caused by the streaming shuffle buffer not adequately randomising 
within the underlying file structure, which appears grouped by label.

Switching to direct dataset loading with `.shuffle()` and `.select()` 
resolved the issue, producing a properly distributed sample across 
all five star ratings (1-star: 20, 2-star: 12, 3-star: 16, 4-star: 24, 
5-star: 8).

## Convert Ratings to Sentiment Labels
Mapping star ratings to sentiment categories using the same logic 
established in our original dataset, ratings 1-2 as negative, 
3 as neutral, 4-5 as positive.

In [10]:
def rating_to_sentiment(rating):
    if rating <= 2:
        return 'negative'
    elif rating == 3:
        return 'neutral'
    else:
        return 'positive'

real_world_df['true_sentiment'] = real_world_df['star_rating'].apply(rating_to_sentiment)

print("Sentiment distribution in real-world sample:")
print(real_world_df['true_sentiment'].value_counts())

Sentiment distribution in real-world sample:
true_sentiment
negative    32
positive    32
neutral     16
Name: count, dtype: int64
